# G04 - Hal Fiyatlarıyla Akıllı Satın Alma: Veri Çekme, Temizleme ve Zenginleştirme (`01_veri_cekme.ipynb`)

- **Kaynak:** İzmir Büyükşehir Belediyesi Açık Veri Portalı – Sebze ve Meyve Hal Fiyatları (CKAN Data API) & TCMB EVDS TÜFE Verisi
- **Çekme Tarihi:** 2026-11-19
- **Lisans:** İzmir Büyükşehir Belediyesi Açık Veri Lisansı
- **Kapsam:** 2006–2025 yılları arası günlük sebze-meyve hal fiyatları (`ASGARI_UCRET`, `AZAMI_UCRET`, `ORTALAMA_UCRET`), ürün/birim standardizasyonu (`İTHAL..` / `ITHAL..` -> `ITHAL`), TÜFE ile reel fiyat hesabı ve gecikmeli (lag) / oynaklık (volatility) özelliklerinin türetilmesi.

## 1. CKAN API ile İzmir Hal Fiyatlarının Çekilmesi (`hal_fiyatlarilkveri.py`)

In [ ]:
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
import requests

def veri_klasoru_bul(baslangic: Path = Path.cwd()) -> Path:
    for aday in [baslangic] + list(baslangic.parents):
        if aday.name == "01_Veri":
            return aday
        hedef = aday / "01_Veri"
        if hedef.exists() and hedef.is_dir():
            return hedef
    hedef = baslangic / "01_Veri"
    hedef.mkdir(parents=True, exist_ok=True)
    return hedef

VERI_KLASORU = veri_klasoru_bul()

KAYNAK = "İzmir Büyükşehir Belediyesi Açık Veri Portalı - Sebze ve Meyve Hal Fiyatları"
CEKME_TARIHI = datetime.now().strftime("%Y-%m-%d")
LISANS = "İzmir Büyükşehir Belediyesi Açık Veri Lisansı"

dataset_id = "39f883f1-5627-4ec0-b424-8f65baf01969"
show_url = "https://acikveri.bizizmir.com/api/3/action/package_show"

resp = requests.get(show_url, params={"id": dataset_id}, timeout=30)
resp.raise_for_status()
resources = resp.json()["result"]["resources"]

csv_resources = [
    r for r in resources if str(r.get("format", "")).upper() == "CSV"
]

parcalar = []
for r in csv_resources:
    ad = r.get("name", "adsiz")
    url = r.get("url")
    try:
        df_parca = pd.read_csv(url, sep=";")
        df_parca["kaynak_dosya"] = ad
        parcalar.append(df_parca)
    except Exception as e:
        print(f"HATA -> {ad} | {e}")

ham_df = pd.concat(parcalar, ignore_index=True)

print(f"Kaynak: {KAYNAK} | Çekme Tarihi: {CEKME_TARIHI} | Lisans: {LISANS}")
print("Düzgün ayrılmış tablo boyutu (satır, sütun):", ham_df.shape)
print("Sütunlar:", list(ham_df.columns))
print("\nİlk 5 satır:")
print(ham_df.head(5))

ham_df.to_csv(VERI_KLASORU / "ham_hal_fiyatlari.csv", index=False, encoding="utf-8-sig")

## 2. Veri Temizleme ve Kategori Standardizasyonu (`hal_fiyat_temizle.py`)
- **`MAL_TURU` İTHAL Birleştirilmesi:** `İTHAL..`, `ITHAL..`, `İTHAL...`, `İTHAL` gibi farklı yazımlar tek bir **`ITHAL`** etiketi altında birleştirilir.
- **Min / Max (`ASGARI_UCRET` ve `AZAMI_UCRET`) Korunması:** Tedarikçi teklif ve fiyat bandı kontrolü için asgari, azami ve ortalama fiyat sütunlarının tamamı sayısal tipe dönüştürülerek korunur.

In [ ]:
import re

def mal_turu_standartlastir(deger: str) -> str:
    s = str(deger).strip().upper()
    s = re.sub(r"\s+", " ", s)
    s_norm = s.replace("İ", "I").strip(".")
    if s_norm.startswith("ITHAL"):
        return "ITHAL"
    return s.strip(".")

def metin_standartlastir(seri: pd.Series) -> pd.Series:
    return (
        seri.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )

df = pd.read_csv(VERI_KLASORU / "ham_hal_fiyatlari.csv", encoding="utf-8-sig", low_memory=False)
df["TARIH"] = pd.to_datetime(df["TARIH"], errors="coerce")
df["MAL_ADI"] = metin_standartlastir(df["MAL_ADI"])
df["MAL_TURU"] = df["MAL_TURU"].apply(mal_turu_standartlastir)
if "BIRIM" in df.columns:
    df["BIRIM"] = metin_standartlastir(df["BIRIM"])

for col in ["ASGARI_UCRET", "AZAMI_UCRET", "ORTALAMA_UCRET"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["TARIH", "MAL_ADI", "ORTALAMA_UCRET"])
df = df[df["ORTALAMA_UCRET"] > 0]
df = df.drop_duplicates()

df.to_csv(VERI_KLASORU / "temiz_hal_fiyatlari.csv", index=False, encoding="utf-8-sig")

ilk_25 = df["MAL_ADI"].value_counts().head(25).reset_index()
ilk_25.columns = ["MAL_ADI", "ADET"]
ilk_25.to_csv(VERI_KLASORU / "ilk_25_urun.csv", index=False, encoding="utf-8-sig")

print("Temiz veri kaydedildi. Boyut:", df.shape)
print("Benzersiz MAL_TURU değerleri:", sorted(df["MAL_TURU"].dropna().unique().tolist()))

## 3. TÜFE ile Birleştirme ve Reel Fiyat Hesabı (`tufe_analizi.py`)
Nominal fiyatlar (`ASGARI_UCRET`, `AZAMI_UCRET`, `ORTALAMA_UCRET`) korunurken TÜFE endeksiyle enflasyondan arındırılmış reel fiyatlar (`REEL_ASGARI_UCRET`, `REEL_AZAMI_UCRET`, `REEL_ORTALAMA_UCRET`) hesaplanır.

In [ ]:
df_25_liste = pd.read_csv(VERI_KLASORU / "ilk_25_urun.csv")
df_tufe = pd.read_csv(VERI_KLASORU / "tufe_temiz.csv")
df_hal = pd.read_csv(VERI_KLASORU / "temiz_hal_fiyatlari.csv", low_memory=False)

df_hal["MAL_ADI"] = metin_standartlastir(df_hal["MAL_ADI"])
df_hal["MAL_TURU"] = df_hal["MAL_TURU"].apply(mal_turu_standartlastir)
if "BIRIM" in df_hal.columns:
    df_hal["BIRIM"] = metin_standartlastir(df_hal["BIRIM"])

secili_urunler = metin_standartlastir(df_25_liste["MAL_ADI"].dropna()).unique()
df_hal_25 = df_hal[df_hal["MAL_ADI"].isin(secili_urunler)].copy()

tarih_dt = pd.to_datetime(df_hal_25["TARIH"], errors="coerce")
df_hal_25["YIL"] = tarih_dt.dt.year
df_hal_25["AY"] = tarih_dt.dt.month

df_birlesik = df_hal_25.merge(df_tufe, on=["YIL", "AY"], how="left")

gecerli_tufe = df_birlesik.dropna(subset=["TUFE"]).sort_values(["YIL", "AY"])
baz_tufe = float(gecerli_tufe["TUFE"].iloc[-1])

df_birlesik["REEL_ORTALAMA_UCRET"] = (
    df_birlesik["ORTALAMA_UCRET"] * (baz_tufe / df_birlesik["TUFE"])
).round(2)

if "ASGARI_UCRET" in df_birlesik.columns:
    df_birlesik["REEL_ASGARI_UCRET"] = (
        df_birlesik["ASGARI_UCRET"] * (baz_tufe / df_birlesik["TUFE"])
    ).round(2)

if "AZAMI_UCRET" in df_birlesik.columns:
    df_birlesik["REEL_AZAMI_UCRET"] = (
        df_birlesik["AZAMI_UCRET"] * (baz_tufe / df_birlesik["TUFE"])
    ).round(2)

df_birlesik.to_csv(VERI_KLASORU / "hal_reel_fiyatli.csv", index=False)

grup_kolonlari = [c for c in ["MAL_TURU", "MAL_ADI", "BIRIM"] if c in df_birlesik.columns]
ozet_25 = (
    df_birlesik.groupby(grup_kolonlari, as_index=False)
    .agg(
        KAYIT=("REEL_ORTALAMA_UCRET", "count"),
        NOMINAL_ORT=("ORTALAMA_UCRET", "mean"),
        REEL_ORT=("REEL_ORTALAMA_UCRET", "mean"),
        REEL_MIN=("REEL_ORTALAMA_UCRET", "min"),
        REEL_MAX=("REEL_ORTALAMA_UCRET", "max"),
    )
    .round(2)
    .sort_values("REEL_ORT", ascending=False)
)

ozet_25.to_csv(VERI_KLASORU / "ilk_25_reel_ozet.csv", index=False)
print("Standartlaştırılmış ilk 25 ürün reel fiyat özeti:")
print(ozet_25.to_string(index=False))

## 4. Özellik Türetme (`ozellik_turetme.py`)
Modelleme ve karar destek arayüzü için `ASGARI_UCRET` (Min) ve `AZAMI_UCRET` (Max) sütunları korunarak gecikmeli değerler (`REEL_LAG_1`, `REEL_LAG_7`, `REEL_LAG_30`), yüzde değişim (`REEL_DEGISIM_YUZDE`) ve oynaklık (`REEL_OYNAKLIK_7`, `REEL_OYNAKLIK_30`) sütunları üretilir ve `01_Veri/hal_reel_ozellikli.csv` olarak kaydedilir.

In [ ]:
df_oz = df_birlesik.copy()
df_oz["TARIH"] = pd.to_datetime(df_oz["TARIH"], errors="coerce")
df_oz = df_oz.sort_values(["MAL_ADI", "BIRIM", "TARIH"]).reset_index(drop=True)

grup = df_oz.groupby(["MAL_ADI", "BIRIM"])["REEL_ORTALAMA_UCRET"]

df_oz["REEL_LAG_1"] = grup.shift(1).round(2)
df_oz["REEL_LAG_7"] = grup.shift(7).round(2)
df_oz["REEL_LAG_30"] = grup.shift(30).round(2)

df_oz["REEL_DEGISIM_YUZDE"] = (grup.pct_change(1) * 100).round(2)
df_oz["REEL_OYNAKLIK_7"] = (
    grup.transform(lambda s: s.rolling(7, min_periods=2).std()).round(2)
)
df_oz["REEL_OYNAKLIK_30"] = (
    grup.transform(lambda s: s.rolling(30, min_periods=2).std()).round(2)
)

# Min (ASGARI_UCRET) ve Max (AZAMI_UCRET) sütunlarının korunduğunu doğrula
zorunlu_sutunlar = ["ASGARI_UCRET", "AZAMI_UCRET", "ORTALAMA_UCRET", "MAL_TURU", "REEL_ORTALAMA_UCRET"]
print("Korunan zorunlu sütunlar:", [c for c in zorunlu_sutunlar if c in df_oz.columns])
print("MAL_TURU kategorileri:", df_oz["MAL_TURU"].value_counts().to_dict())

df_oz.to_csv(VERI_KLASORU / "hal_reel_ozellikli.csv", index=False)
print("hal_reel_ozellikli.csv kaydedildi. Boyut:", df_oz.shape)

## 5. Örnek Veri (`ornek_veri.csv`) ve Veri Sözlüğü (`veri_sozlugu.xlsx`) Oluşturma

In [ ]:
# Örnek veri (İlk 100 satır)
ornek_yolu = VERI_KLASORU / "ornek_veri.csv"
df_oz.head(100).to_csv(ornek_yolu, index=False)
print("Örnek veri kaydedildi:", ornek_yolu)

# Veri sözlüğü
aciklamalar = {
    "TARIH": "Fiyat kaydının ait olduğu tarih bilgisi.",
    "MAL_TURU": "Ürünün ana kategori/tür bilgisi (MEYVE, SEBZE, ITHAL - birleştirilmiş standart büyük harf).",
    "MAL_ADI": "Haldeki ürün adı (standartlaştırılmış büyük harf).",
    "BIRIM": "Ürünün fiyatlandırma birimi (KG, ADET, BAĞ vb.).",
    "ASGARI_UCRET": "Günün en düşük (Min) nominal hal fiyatı (tedarikçi teklif/fiyat bandı kontrolü için korunmuştur).",
    "AZAMI_UCRET": "Günün en yüksek (Max) nominal hal fiyatı (tedarikçi teklif/fiyat bandı kontrolü için korunmuştur).",
    "ORTALAMA_UCRET": "Günün ortalama nominal hal fiyatı.",
    "kaynak_dosya": "Verinin çekildiği yıllık CKAN kaynak dosyası adı.",
    "YIL": "Tarih bilgisinden çıkarılan yıl değeri.",
    "AY": "Tarih bilgisinden çıkarılan ay numarası (1-12).",
    "TUFE": "İlgili yıl ve aya ait Tüketici Fiyat Endeksi değeri.",
    "REEL_ORTALAMA_UCRET": "TÜFE ile enflasyondan arındırılmış reel ortalama fiyat.",
    "REEL_ASGARI_UCRET": "TÜFE ile enflasyondan arındırılmış reel asgari (Min) fiyat.",
    "REEL_AZAMI_UCRET": "TÜFE ile enflasyondan arındırılmış reel azami (Max) fiyat.",
    "REEL_LAG_1": "Aynı ürün ve birimde bir önceki kayıttaki reel ortalama fiyat (1 dönem gecikme).",
    "REEL_LAG_7": "Aynı ürün ve birimde 7 kayıt önceki reel ortalama fiyat (7 dönem gecikme).",
    "REEL_LAG_30": "Aynı ürün ve birimde 30 kayıt önceki reel ortalama fiyat (30 dönem gecikme).",
    "REEL_DEGISIM_YUZDE": "Bir önceki kayda göre reel fiyattaki yüzde değişim oranı.",
    "REEL_OYNAKLIK_7": "Aynı ürün ve birimde son 7 kayıttaki reel fiyatın standart sapması (kısa dönem oynaklık).",
    "REEL_OYNAKLIK_30": "Aynı ürün ve birimde son 30 kayıttaki reel fiyatın standart sapması (orta dönem oynaklık).",
}

satirlar = []
for sutun in df_oz.columns:
    ornek = df_oz[sutun].dropna().iloc[0] if not df_oz[sutun].dropna().empty else np.nan
    satirlar.append({
        "DEGISKEN_ADI": sutun,
        "VERI_TIPI": str(df_oz[sutun].dtype),
        "EKSIK_DEGER_SAYISI": int(df_oz[sutun].isna().sum()),
        "ORNEK_DEGER": ornek,
        "ACIKLAMA": aciklamalar.get(sutun, "Analiz sürecinde kullanılan veya türetilen değişken."),
    })

sozluk_df = pd.DataFrame(satirlar)
xlsx_yolu = VERI_KLASORU / "veri_sozlugu.xlsx"
try:
    sozluk_df.to_excel(xlsx_yolu, index=False)
    print("Veri sözlüğü kaydedildi:", xlsx_yolu)
except ModuleNotFoundError:
    csv_yedek = VERI_KLASORU / "veri_sozlugu.csv"
    sozluk_df.to_csv(csv_yedek, index=False)
    print("openpyxl bulunamadı, CSV kaydedildi:", csv_yedek)